# DSA 405 P2 — House PTR PDF extraction and cleaning audit

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StrokeOfLuck/dsa405-part-2/blob/main/notebooks/DSA405_002_FA26_P2_sryan3.ipynb)

**What changes from P1:** Start with the archived **2025 House filing PDFs**. The original scraper's pinned Stage 3 turns those PDFs into a transaction CSV; Stage 4 resolves ticker and asset fields into a second CSV. This P2 notebook audits Stage 3 and logs the Stage 4 decisions. The published scraper repo is never modified.

**Unit:** one transaction row extracted from a PDF. `source_year=2025` is the filing index year, not necessarily the transaction year.

## 1. Make an isolated 2025 copy and rebuild the CSVs

The pinned source commit and checksums in `data/raw/2025_pdf_manifest.csv` identify 515 PDFs. The committed manifest stays unmodified in `data/raw/`. The script copies and verifies the PDFs in `data/work/01_pdfs/2025` and copies the 2025 XML index into the working directory. The large PDF copies are downloaded on first run; later runs reuse them. This step requires Git and network access the first time. It may take several minutes on CPU.

**What to look for:** `Pipeline result: 0` means the script finished. On the first Colab run it downloads the PDF copies; later runs can reuse verified copies. If it fails, open `data/work/rebuild.log` for the full error. Nothing in this cell writes to the original scraper repository.

In [ ]:
# Path names project files. os changes folders, subprocess runs commands, and sys selects this notebook's Python.
from pathlib import Path
import os, subprocess, sys

# Colab opens the notebook file without cloning the rest of the repository.
# A local session may already be at the project root; Colab initially has only the notebook.
if not Path("scripts/rebuild_2025.py").exists():
    # Jupyter may start inside notebooks/, so move up if the script is there.
    if Path("../scripts/rebuild_2025.py").exists():
        os.chdir("..")
    else:
        # Otherwise Colab needs the whole project, including scripts, manifest, and requirements.
        target = Path("dsa405-part-2")
        if not (target / "scripts/rebuild_2025.py").exists():
            # Clone once; check=True stops execution if Git fails.
            subprocess.run(["git","clone","--depth","1","https://github.com/StrokeOfLuck/dsa405-part-2.git",str(target)],check=True)
        os.chdir(target)

# Install declared dependencies automatically in Colab; a local environment can manage them itself.
try:
    import google.colab  # available in Colab, absent in a normal local notebook
except ImportError:
    pass
else:
    # Use the Python interpreter running this notebook for Stage 3 and Stage 4 packages.
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"],check=True)

# Keep verbose pipeline output in a reproducible log; create its folder first.
log_path=Path("data/work/rebuild.log")
log_path.parent.mkdir(parents=True,exist_ok=True)
# Copy and verify 2025 PDFs, then build the two CSVs; capture output and errors in the log.
with log_path.open("w") as log:
    result=subprocess.run([sys.executable,"scripts/rebuild_2025.py"],stdout=log,stderr=subprocess.STDOUT)
# Show the exit status and last 18 log lines without filling the notebook with pipeline output.
print("Pipeline result:",result.returncode)
print("\n".join(log_path.read_text(errors="replace").splitlines()[-18:]))
# Stop later audit cells from using stale or partial CSVs after a failed rebuild.
if result.returncode:
    raise RuntimeError(f"Pipeline failed. Inspect {log_path} for the stage and error.")

The pinned original PDFs remain untouched in the scraper archive; the P2 working copies are checked against the committed manifest. Stage 3 writes `transactions_raw.csv`; Stage 4 writes `transactions_resolved.csv`. Publication makes a web-facing CSV too. No source PDF is changed.

## 2. Audit the Stage 3 transaction CSV

**What to look for:** the expected rebuild has 515 copied PDFs and 7,667 transaction rows in each CSV. Counts describe this pinned snapshot; investigate if a future input changes them.

In [ ]:
# pandas compares transaction tables; display renders readable tables in Colab/Jupyter.
import pandas as pd
from IPython.display import display

# Name the first parser output (Stage 3) and later ticker resolver output (Stage 4).
source=Path("data/work/04_transactions/transactions_raw.csv")
resolved=Path("data/work/04_transactions/transactions_resolved.csv")
# Read all columns as text to preserve IDs and source wording; keep blanks as empty strings for consistent missing counts.
raw=pd.read_csv(source,dtype=str,keep_default_na=False)
clean=pd.read_csv(resolved,dtype=str,keep_default_na=False)
# Confirm row and column counts before comparing stages.
print("V8.1 extracted:",raw.shape,"V8.2 resolved:",clean.shape)
# Compare copied PDFs with the 515-file manifest; source_year is the filing index year, not necessarily trade year.
print("Source PDFs copied:",len(list(Path("data/work/01_pdfs/2025").glob("*.pdf"))))
print("Source year:",raw.source_year.value_counts(dropna=False).to_dict())
# Inspect five transactions with identifiers, parsed fields, and review flags.
display(raw[["filing_id","transaction_number_in_filing","politician","asset","ticker","transaction_date","amount_category","needs_review"]].head(5))

### Column inventory

Read identifiers as strings to preserve their printed form. Empty strings count as missing. Audit every field used in this notebook before looking at the resolved output.

**What to look for:** a missing value here is exactly `""` in the CSV; it does not by itself mean the underlying PDF lacked the information.

In [ ]:
# Audit these 19 fields: identifiers, normalized values, original PDF text, provenance, and review flags.
FIELDS=["filing_id","transaction_number_in_filing","politician","source_year","owner","asset","ticker","asset_type","transaction_type","transaction_date","amount_min","amount_max","amount_status","needs_review","review_reason","original_pdf_url","asset_raw","transaction_date_raw","amount_raw"]
# CSV loading protected IDs by treating everything as text; document intended analytic types separately.
TYPES={"filing_id":"string ID","transaction_number_in_filing":"integer","politician":"string","source_year":"integer","owner":"category","asset":"string","ticker":"string","asset_type":"category","transaction_type":"category","transaction_date":"date","amount_min":"USD lower bound","amount_max":"USD upper bound","amount_status":"category","needs_review":"boolean","review_reason":"string","original_pdf_url":"URL","asset_raw":"source text","transaction_date_raw":"source text","amount_raw":"source text"}
# Fail if the parser schema changed instead of producing a misleading inventory.
assert set(FIELDS)<=set(raw.columns)
# Count exactly empty strings as missing and report their share (four decimals) and distinct values.
# A blank parser field does not prove the source PDF lacked the information.
audit=pd.DataFrame([{"variable":x,"loaded dtype":str(raw[x].dtype),"intended type":TYPES[x],"missing count":int(raw[x].eq("").sum()),"missing rate":round(raw[x].eq("").mean(),4),"distinct including blank":int(raw[x].nunique(dropna=False))} for x in FIELDS])
# Render the complete variable inventory.
display(audit)

**What to look for:** `transaction_date_raw` can fail strict standalone-date parsing because neighboring PDF text was captured; inspect examples in the next cell before calling the normalized date incorrect.

In [ ]:
# Print all observed levels only for fields with fewer than 30 values; avoid dumping free text.
for field in FIELDS:
    if raw[field].nunique(dropna=False)<30:
        print(field,raw[field].value_counts(dropna=False).to_dict())
# Numeric ranges apply to transaction position and dollar bounds, not source text.
for field in ["transaction_number_in_filing","amount_min","amount_max"]:
# Turn blanks into missing values and coerce malformed numbers without changing the original CSV.
    parsed=pd.to_numeric(raw[field].replace("",pd.NA),errors="coerce")
# Show min, max, span, and nonblank values that failed numeric parsing.
    print(field,"min",parsed.min(),"max",parsed.max(),"range",parsed.max()-parsed.min(),"nonempty invalid",int((parsed.isna()&raw[field].ne("")).sum()))
# Normalized dates use ISO format; raw PDF text is expected to be a standalone slashed US date.
for field,fmt in [("transaction_date","%Y-%m-%d"),("transaction_date_raw","%m/%d/%Y")]:
# Strict parsing surfaces invalid dates and text attached to a raw date.
    parsed=pd.to_datetime(raw[field].replace("",pd.NA),format=fmt,errors="coerce")
# Report valid date range and count nonblank unparsed values for each field.
    print(field,"min",parsed.min(),"max",parsed.max(),"nonempty unparsed",int((parsed.isna()&raw[field].ne("")).sum()))

**What to look for:** 640 raw-date strings fail strict standalone parsing in this snapshot; 617 were unflagged. This check identifies an extraction-text issue; it does not prove that the normalized transaction date is wrong.

In [ ]:
# A filing can have many transaction rows; filing ID plus row position is the proposed key.
keys=["filing_id","transaction_number_in_filing"]
# Exact duplicates inflate totals; repeated keys may indicate distinct records assigned one identifier.
checks={
 "exact duplicates":int(raw.duplicated().sum()),
 "duplicate filing+row keys":int(raw.duplicated(keys).sum()),
# Placeholder strings can disguise missing values; trim and lowercase before counting.
 "sentinel tokens":int(sum(raw[x].str.strip().str.lower().isin(["n/a","na","null","none","999"]).sum() for x in FIELDS)),
# The Unicode replacement character can point to damaged PDF text encoding.
 "replacement character (encoding)":int(sum(raw[x].str.contains("\ufffd",regex=False).sum() for x in FIELDS)),
# Leading-zero numeric-looking IDs must remain strings when CSVs are loaded.
 "leading-zero IDs":int(raw.filing_id.str.match(r"^0[0-9]+$").sum()),
# An extracted PDF table total could otherwise be mistaken for a transaction.
 "total/subtotal labels":int(sum(raw[x].str.contains(r"^\s*(?:total|subtotal)\s*$",case=False,regex=True).sum() for x in ["asset","politician"])),
# A lower dollar bound above the upper bound is inconsistent; coerce values only for this comparison.
 "nonempty amount min > max":int((pd.to_numeric(raw.amount_min,errors="coerce")>pd.to_numeric(raw.amount_max,errors="coerce")).sum()),
# A parser review flag requests a source-PDF check; it does not establish an error.
 "review flagged":int(raw.needs_review.eq("True").sum()),
}
# Display each named integrity count.
display(pd.Series(checks,name="count").to_frame())
# Require the whole raw field to parse as MM/DD/YYYY, separately from the normalized date.
raw_date=pd.to_datetime(raw.transaction_date_raw.replace("",pd.NA),format="%m/%d/%Y",errors="coerce")
# Mark nonempty raw strings that fail that strict parse, including attached PDF text.
bad_date=raw.transaction_date_raw.ne("")&raw_date.isna()
# Count exceptions and those Stage 3 left unflagged.
print("Non-standalone raw date strings:",int(bad_date.sum()),"of which scraper did not flag:",int((bad_date&raw.needs_review.eq("False")).sum()))
# Show eight identifiers, both dates, review status, and PDF links for manual inspection.
display(raw.loc[bad_date,["filing_id","transaction_number_in_filing","transaction_date_raw","transaction_date","needs_review","original_pdf_url"]].head(8))

**Check the exceptions:** Text beside a date may have bled in from an adjacent PDF cell. This does not automatically make the resolved ISO date wrong. Open the linked original PDF for examples before changing them. Explicitly record defect classes checked and not found above.

## 3. Data dictionary for Stage 3 fields

**What to look for:** observed category levels come from this CSV, while the notes and expected ranges are interpretations to verify against the disclosure and parser code.

In [ ]:
# Record the meaning and caveat of every audited field: source_year is an index year;
# amount bounds describe disclosed bands rather than exact trade sizes.
NOTES={
 "filing_id":"House filing ID; keep as text, not a person identifier", "transaction_number_in_filing":"Row position in one filing; use with filing_id as proposed key", "politician":"Reported member name; not a stable ID", "source_year":"Index year, not necessarily trade year", "owner":"May be blank if no separate owner given", "asset":"V8.1 parser text; V8.2 preserves as asset_v8_1", "ticker":"V8.1 parsed candidate; V8.2 preserves as ticker_v8_1", "asset_type":"Source disclosure asset class", "transaction_type":"P purchase, S sale, E exchange; partial sales retain suffix", "transaction_date":"Reported trade date, not filing date", "amount_min":"Minimum of disclosed band, not an exact amount", "amount_max":"Maximum of disclosed band; blanks may mean open ended or uncertain", "amount_status":"Parser's amount classification", "needs_review":"Parser-generated flag, not proof that unflagged rows are correct", "review_reason":"Parser reasons; may be blank when not flagged", "original_pdf_url":"Link to the official source PDF", "asset_raw":"Text from source extraction; retain for reversibility", "transaction_date_raw":"Source-like text can contain neighboring PDF fragments", "amount_raw":"Source-like amount text; preserve alongside numeric bounds"
}
# State defensible expected values only; other fields are source-defined.
RANGES={"source_year":"2025","transaction_number_in_filing":"positive integer","transaction_date":"valid calendar date","amount_min":"nonnegative USD","amount_max":"nonnegative USD or blank for nonstandard cases","needs_review":"True / False"}
# For each field, show type, units, observed small-category levels, expected range, blanks,
# missing-value interpretation, and the note above. Observed levels come from this CSV.
dictionary=pd.DataFrame([{"variable":x,"type":TYPES[x],"units":"USD" if x in ["amount_min","amount_max"] else ("calendar date" if x=="transaction_date" else "code/text or none"),"factor levels":", ".join(sorted(raw[x].unique())) if raw[x].nunique(dropna=False)<30 else "—", "valid range":RANGES.get(x,"Source-defined; check PDF"),"missing count":int(raw[x].eq("").sum()),"missing means":"Blank in parser output; confirm against PDF" if raw[x].eq("").any() else "None in this extract", "notes":NOTES[x]} for x in FIELDS])
# Render the dictionary for use in the assignment and source-PDF review.
display(dictionary)

## 4. Quantified extraction and cleaning decisions

Stage 3 extracts values from PDF text and writes V8.1. Stage 4 resolves ticker and display-asset fields to V8.2. This log counts each decision **on the 2025 data** and names the original evidence that lets a reader reverse it. The source code for both stages is pinned in the manifest. The class-specific check adds a flag for raw date text that includes neighboring content, without changing the reported transaction date.

This cell records transformations **with a count, reason, information loss, and reversal path**. Counts from different decisions can overlap; do not add them as a number of unique transactions.

**What to look for:** in the pinned batch the six direct Stage 4 comparisons are all zero. The date-prefix disagreement count is zero too, despite extra text in some raw date strings. The classification decisions still matter; inspect the original PDFs before making a manual correction.

In [ ]:
# A row-by-row value comparison requires equal counts, identical key order, and unique filing/row keys.
assert len(raw)==len(clean),"Stage 4 unexpectedly changed row count"
assert raw[keys].reset_index(drop=True).equals(clean[keys].reset_index(drop=True)),"Stage 4 changed transaction order or keys"
assert raw.duplicated(keys).sum()==0,"Resolve duplicate keys before accepting one-to-one comparison"
# Build an audit log; each decision records its stage, count, reason, information loss, and reversal path.
changes=[]
def log_decision(stage,column,action,count,reason,lost,reverse):
# Number each decision consistently so the table can be cited and reviewed.
    changes.append({"#":len(changes)+1,"stage":stage,"column":column,"change made":action,"rows/cells affected":int(count),"why":reason,"what is lost":lost,"how to reverse":reverse})

# Stage 3: compare the preserved extraction text with its parsed field.
# Compare three Stage 3 raw fields with their parsed analytic versions: asset, transaction type, and date.
for original,parsed,reason in [
 ("asset_raw","asset","Separate the displayed asset from ticker and nearby PDF text for analysis"),
 ("transaction_type_raw","transaction_type","Isolate the P/S/E transaction code from adjacent PDF text"),
 ("transaction_date_raw","transaction_date","Convert source-like date text to an ISO date for chronological checks")]:
# Count changed representations; retain source-like text in the same transaction row.
    count=int(raw[original].ne(raw[parsed]).sum())
    log_decision("3: PDF extraction",parsed,f"Parsed {original} into {parsed}",count,reason,f"Nothing from {original}; source-like text remains in the same row.",f"Re-read {original} and inspect original_pdf_url; re-run pinned Stage 3.")
# Count nonblank ticker candidates inferred from asset text, with raw text and PDF still available.
log_decision("3: PDF extraction","ticker","Extracted ticker candidates from asset text",int(raw.ticker.ne("").sum()),"Source PDF asset text can contain a parenthetical stock symbol.","No raw text lost; asset_raw remains alongside ticker.","Compare ticker to asset_raw and the original PDF; re-run Stage 3.")
# Count each recognized, nonstandard, or uncertain amount classification; retain amount_raw.
for status,count in raw.amount_status.value_counts(dropna=False).items():
    reasons={"valid_range":"Two source dollar values match a recognized disclosure band.","nonstandard_exact":"The source displays a nonstandard exact amount rather than a standard band.","missing_range_bound":"The source range cannot safely supply both numeric bounds."}
    log_decision("3: PDF extraction","amount_status",f"Classified amount as {status}",count,reasons.get(status,"Amount parser classified source text."),"Raw amount text remains in amount_raw; numeric bounds may be absent when uncertain.","Compare amount_raw, amount_min, amount_max and the source PDF; re-run Stage 3.")

# Stage 4: log classifications, and quantify actual changed values separately.
# Stage 4 can classify a ticker without changing the visible value; log each status separately.
for status,count in clean.ticker_parse_status.value_counts(dropna=False).items():
    reasons={"accepted":"A source-structured ticker candidate was accepted.","not_applicable":"The row is not a stock asset, so stock ticker resolution does not apply.","ambiguous_preserved":"The candidate is too uncertain to assert as a ticker."}
    log_decision("4: ticker resolver","ticker_parse_status",f"Classified ticker as {status}",count,reasons.get(status,"Resolver classification; inspect pinned Stage 4 code."),"No source evidence lost; V8.1 ticker and asset are retained.","Use ticker_v8_1, asset_v8_1, and asset_raw or re-run pinned Stage 4.")
# Compare six resolved fields with the V8.1 copies Stage 4 retained alongside them.
actual_changes={current:int(clean[current].ne(clean[prior]).sum()) for current,prior in [
 ("ticker_v8_2_cleaned","ticker_v8_1"),("asset_v8_2_cleaned","asset_v8_1"),
 ("review_reason","review_reason_v8_1"),("review_level","review_level_v8_1"),
 ("needs_review","needs_review_v8_1"),("geometry_quality_score","geometry_quality_score_v8_1")]}
# Report actual replacements, including zero counts.
print("Stage 4 values changed (zero is a legitimate finding):",actual_changes)
# Only log a value replacement when at least one value changed; V8.1 columns allow reversal.
for field,count in actual_changes.items():
    if count:
        prior={"ticker_v8_2_cleaned":"ticker_v8_1","asset_v8_2_cleaned":"asset_v8_1","review_reason":"review_reason_v8_1","review_level":"review_level_v8_1","needs_review":"needs_review_v8_1","geometry_quality_score":"geometry_quality_score_v8_1"}[field]
        log_decision("4: ticker resolver",field,"Replaced V8.1 value",count,"Resolver accepted a candidate or recalculated a ticker-specific review flag.",f"Nothing; {prior} retains the previous value.",f"Restore directly from {prior}.")
# Six preserved fields add len(clean) × 6 cells; this is not a count of unique rows.
log_decision("4: ticker resolver","six V8.1 audit columns","Preserved original fields next to resolved values",len(clean)*6,"Readers need a reversible before/after record.","Nothing; columns are added copies.","Drop the six *_v8_1 columns to reverse this addition.")

# P2-specific audit flags: distinguish source text contamination from a wrong date.
# Keep loaded Stage 4 data intact; add three audit fields to a separate P2 copy.
p2=clean.copy()
source_date=p2.transaction_date_raw
# Flag raw date text that is not exactly a slashed date; leave normalized dates unchanged.
standalone=source_date.str.fullmatch(r"\d{1,2}/\d{1,2}/\d{4}")
p2["raw_date_has_extra_text"]=~standalone
# Capture a leading date token even when trailing PDF text is attached.
p2["raw_date_prefix"]=source_date.str.extract(r"^(\d{1,2}/\d{1,2}/\d{4})",expand=False).fillna("")
# Parse that token to ISO; blanks and invalid tokens stay empty.
parsed_prefix=pd.to_datetime(p2.raw_date_prefix.replace("",pd.NA),format="%m/%d/%Y",errors="coerce").dt.strftime("%Y-%m-%d").fillna("")
# A valid leading token that differs from the normalized transaction date needs PDF review.
p2["date_prefix_disagrees"]=parsed_prefix.ne("")&parsed_prefix.ne(p2.transaction_date)
# Record counts, rationale, and reversal instructions for the three new audit columns.
log_decision("P2: audit","raw_date_has_extra_text","Flagged non-standalone source date text",int(p2.raw_date_has_extra_text.sum()),"Adjacent PDF text may be attached to a date and needs source review.","Nothing; original transaction_date_raw remains unchanged.","Remove the flag; compare original text with the source PDF.")
log_decision("P2: audit","raw_date_prefix","Extracted leading date token for checking",int(p2.raw_date_prefix.ne("").sum()),"Check whether the V8.1 date agrees with the visible leading date despite extra text.","Nothing; source-like text remains in transaction_date_raw.","Drop this derived column; extract again from transaction_date_raw.")
log_decision("P2: audit","date_prefix_disagrees","Flagged parsed date disagreements",int(p2.date_prefix_disagrees.sum()),"Any disagreement between source-like date prefix and normalized date warrants source review.","Nothing; both original and normalized dates remain.","Remove the derived flag and recompute from the two date fields.")
# Save the final P2 CSV in data/clean, separate from source files and intermediate CSVs.
output=Path("data/clean/house_ptr_2025_p2.csv")
output.parent.mkdir(parents=True,exist_ok=True)
# Avoid writing an artificial pandas index column.
p2.to_csv(output,index=False)
# Display the audit log and the final path and date-check counts.
log_df=pd.DataFrame(changes)
display(log_df)
print("P2 clean output:",output)
print("Raw date extra-text flags:",int(p2.raw_date_has_extra_text.sum()),"date-prefix disagreements:",int(p2.date_prefix_disagrees.sum()))

The final P2 CSV also contains three audit fields: `raw_date_has_extra_text` (boolean; true when the raw text is not a standalone date), `raw_date_prefix` (text date in MM/DD/YYYY form extracted from the start, blank if absent), and `date_prefix_disagrees` (boolean; true if that prefix differs from the parsed transaction date). Each uses the raw text, so the source remains recoverable.

Stage 4 made zero changes to the six compared value fields in this 2025 batch; its ticker classifications and audit columns are still documented above. Stage 3 did make substantive parsing decisions, including the PDF text fields shown in the log. The P2 flags are review aids, not claims that the already parsed transaction date is wrong. Inspect source PDFs and add a numbered decision for any manual correction.

## 5. Row and column accounting

**What to look for:** 7,667 rows remain at each step; the P2 CSV has 64 columns. The displayed “removed/renamed” and “added/renamed” numbers describe name differences, not evidence that raw PDF data was deleted.

In [ ]:
# Compare field names across Stage 3 and P2; renamed fields are not necessarily lost data.
removed=sorted(set(raw.columns)-set(p2.columns))
added=sorted(set(p2.columns)-set(raw.columns))
# Reconcile starting/ending rows and columns; this notebook filters no transactions,
# so exact, near-duplicate, and other removal counts are zero.
accounting=pd.DataFrame([("Raw V8.1 rows",len(raw)),("Exact duplicate rows removed",0),("Near-duplicate rows removed",0),("Other rows removed",0),("P2 clean rows",len(p2)),("Raw columns",raw.shape[1]),("Columns removed/renamed",len(removed)),("Columns added/renamed",len(added)),("P2 clean columns",p2.shape[1])],columns=["item","count"])
# Catch accidental row changes and verify the field-name arithmetic.
assert len(raw)==len(p2)
assert raw.shape[1]-len(removed)+len(added)==p2.shape[1]
# Display counts, changed names, and the exact paths of all three CSV stages.
display(accounting)
print("Removed or renamed:",removed)
print("Added or renamed:",added)
print("Raw V8.1 CSV:",source)
print("V8.2 CSV:",resolved)
print("P2 clean CSV:",output)

## 6. Provenance brief (under 200 words)

This dataset comes from 515 periodic transaction report PDFs in the U.S. House Clerk's 2025 disclosure archive. Members submit the reports to disclose transactions, and the House publishes the PDFs. This project copies the PDFs from a pinned archive of my House PTR scraper without modifying the archive. The same parser extracts transaction rows into a first CSV; its next stage resolves some ticker and asset names into a second CSV. Each row retains a link to its source disclosure, and uncertain cases can be checked against that PDF.

The dataset describes **reported transactions in filings indexed under 2025**, even when a transaction happened in an earlier year. Disclosed amounts are often ranges, not exact trade values. The parser can misread a PDF or miss a transaction; a row without a review flag is not independently verified. It cannot establish unreported trades or prove that a member personally executed a trade.

**Before submission:** Review example PDFs and adjust this brief if audit findings change the limits.

## Submission check

- [ ] Inspect representative review cases and raw-date exceptions against their linked PDFs.
- [ ] Record any manual decisions with exact affected counts, reasons, losses and reversals.
- [ ] Run notebook from a restarted kernel, save executed outputs, and verify the counts.
- [ ] Include the course's self-scored rubric with notebook and repo link.